# NB-03: Training

**Purpose:** Fine-tune MobileMamba-B1 on the UCF-Crime frame dataset.

**Strategy:**
- Freeze backbone for first 5 epochs (train head only)
- Unfreeze and train with differential learning rates
- Class-weighted CrossEntropyLoss to handle imbalance

> **For long runs:** Use `python src/trainer.py` from terminal to avoid kernel timeouts.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../..'))

%load_ext autoreload
%autoreload 2


import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
# ─── Training Config (FINAL — All 7 Bugs Fixed) ───────
config = {
    'data_dir': '../data/frames',
    'checkpoint_dir': '../checkpoints',
    'log_dir': '../outputs/logs',
    'pretrained_path': '../checkpoints/best_model.pth',
    'batch_size': 8,            # Lowered for temporal clips (8 clips x 8 frames = 64 imgs)
    'epochs': 60,
    'lr_backbone': 1e-6,        # Very gentle for partial unfreeze
    'lr_head': 1e-4,
    'weight_decay': 0.05,
    'freeze_epochs': 0,        # Stabilize head before unfreezing Stage 3
    'grad_clip': 1.0,
    'img_size': 256,
    'num_workers': 4,
    'patience': 12,
    'warmup_epochs': 5,
    'mixup_alpha': 0.2,
    'temporal_size': 8,         # Enable 8-frame clips
}



print('Training config (FINAL):')
for k, v in config.items():
    print(f'  {k}: {v}')


Training config (FINAL):
  data_dir: ../data/frames
  checkpoint_dir: ../checkpoints
  log_dir: ../outputs/logs
  pretrained_path: ../checkpoints/best_model.pth
  batch_size: 8
  epochs: 60
  lr_backbone: 1e-06
  lr_head: 0.0001
  weight_decay: 0.05
  freeze_epochs: 0
  grad_clip: 1.0
  img_size: 256
  num_workers: 4
  patience: 12
  warmup_epochs: 5
  mixup_alpha: 0.2
  temporal_size: 8


In [ ]:
# ─── Run Training ─────────────────────────────────────
from src.trainer import AnomalyTrainer

trainer = AnomalyTrainer(config)
history = trainer.train()

c:\Users\sunny\OneDrive\Desktop\MobilMamba\mm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Can not import selective_scan_cuda_oflex. This affects speed.


c:\Users\sunny\OneDrive\Desktop\MobilMamba\mm\venv\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
c:\Users\sunny\OneDrive\Desktop\MobilMamba\mm\model\lib_mamba\csm_triton.py:11: UserWarning: Triton not installed, fall back to pytorch implements. (Error: No module named 'triton')
  warnings.warn(f"Triton not installed, fall back to pytorch implements. (Error: {e})")
c:\Users\sunny\OneDrive\Desktop\MobilMamba\mm\model\lib_mamba\csms6s.py:13: UserWarning: Can not import selective_scan_cuda_oflex. This affects speed.
  warnings.warn("Can not import selective_scan_cuda_oflex. This affects speed.")
c:\Users\sunny\OneDrive\Desktop\MobilMamba\mm\model\lib_mamba\csms6s.py:73: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_ty

[Trainer] Using device: cuda
[TRAIN] Loaded 280 clips (T=8)
[VAL] Loaded 64 clips (T=8)
[Trainer] WeightedRandomSampler: {0: 35, 1: 35, 2: 105, 3: 35, 4: 35, 5: 35}
[Pretrained] Full Model loaded from ../checkpoints/best_model.pth (Flexible Mode)
  Note: 6 new layers initialized fresh (e.g. BatchNorm/ReLU)
[Trainer] Class weights: [1.3333333730697632, 1.3333333730697632, 0.4444444477558136, 1.3333333730697632, 1.3333333730697632, 1.3333333730697632]

  MobileMamba Anomaly Detection — Training
  Epochs: 60  |  Batch: 8
  Backbone LR: 1e-06  |  Head LR: 0.0001
  Freeze epochs: 0
  Early stopping patience: 12



Epoch 1/60 [TRAIN]:  66%|██████▌   | 23/35 [20:52<11:12, 56.02s/it, acc=16.3%, loss=1.8259]

In [ ]:
# ─── Plot Training Progress ───────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Loss curves
ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(epochs, [a * 100 for a in history['train_acc']], 'b-', label='Train Acc')
ax2.plot(epochs, [a * 100 for a in history['val_acc']], 'r-', label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/logs/training_curves.png', dpi=150)
plt.show()

print(f'\nBest validation accuracy: {max(history["val_acc"]) * 100:.1f}%')